# Notebook 04: GARCH(1,1) Benchmark

The GARCH(1,1) model (Generalized Autoregressive Conditional Heteroskedasticity) is the
classical benchmark for modeling volatility clustering in financial return series.
Unlike constant-volatility models, GARCH captures the empirical fact that large shocks
tend to be followed by large shocks (of either sign), producing time-varying conditional variance:

$$\sigma^2_t = \omega + \alpha\,(r_{t-1} - \mu)^2 + \beta\,\sigma^2_{t-1}$$

This notebook fits a GARCH(1,1) to observed equity returns via maximum likelihood,
simulates synthetic paths, and evaluates the same validation metrics used for the CHMM.
The purpose is to establish a single-regime benchmark against which the regime-switching
CHMM can be compared.

### Outputs
- Fitted GARCH parameters and conditional volatility plot
- Validation metrics (KS, kurtosis, ACF-MAE, Wasserstein, Hellinger)
- Three-panel comparison figure (density, ACF, Q-Q)

## Setup

In [ ]:
include("../Include.jl");

## Configuration (Tunable)

In [ ]:
# --- TUNABLE PARAMETERS ---
ticker = "SPY";
N_PATHS = 1000;             # Monte Carlo paths for simulation
L = 252;                    # ACF max lag (1 trading year)
risk_free_rate = 0.0;
Δt = 1/252;

## Load Data and Compute Returns

In [ ]:
# --- LOAD DATA ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

list_of_all_tickers = keys(dataset) |> collect |> sort;
all_firms_excess_return_matrix = log_growth_matrix(dataset, list_of_all_tickers; Δt=Δt, risk_free_rate=risk_free_rate);
Rᵢ = findfirst(x -> x == ticker, list_of_all_tickers) |> i -> all_firms_excess_return_matrix[:, i];
R_is = Rᵢ[1:(maximum_number_trading_days-1)];

println("Training data: $(length(R_is)) observations for $ticker");

## Fit GARCH(1,1)

The model is fit via grid-initialized Nelder-Mead MLE (no external optimizer package).
The `build` factory returns a `MyGARCHModel` with fitted parameters and the conditional
variance history.

In [ ]:
# --- FIT GARCH(1,1) ---
garch_model = build(MyGARCHModel, (observations=R_is,));
println("GARCH(1,1) fitted for $ticker");

## Print Parameters

In [ ]:
# --- PARAMETER SUMMARY ---
persistence = garch_model.α + garch_model.β;
uncond_var = garch_model.ω / max(1.0 - persistence, 1e-8);
uncond_vol = sqrt(uncond_var);

println("╔══════════════════════════════════════════════════════╗");
println("║  GARCH(1,1) Parameters — $ticker                     ║");
println("╠══════════════════════════════════════════════════════╣");
println("║  ω (intercept):       $(lpad(round(garch_model.ω, digits=6), 12))               ║");
println("║  α (ARCH coeff):      $(lpad(round(garch_model.α, digits=6), 12))               ║");
println("║  β (GARCH coeff):     $(lpad(round(garch_model.β, digits=6), 12))               ║");
println("║  μ (mean):            $(lpad(round(garch_model.μ, digits=6), 12))               ║");
println("║  α + β (persistence): $(lpad(round(persistence, digits=6), 12))               ║");
println("║  Unconditional vol:   $(lpad(round(uncond_vol, digits=4), 12))               ║");
println("║  Log-likelihood:      $(lpad(round(garch_model.log_likelihood, digits=2), 12))               ║");
println("╚══════════════════════════════════════════════════════╝");

## Conditional Variance Plot

The fitted conditional standard deviation $\sqrt{\sigma^2_t}$ is overlaid on the absolute
returns $|r_t|$. A well-fitted GARCH tracks the envelope of volatility clusters.

In [ ]:
# --- CONDITIONAL VOLATILITY PLOT ---
cond_vol = sqrt.(garch_model.σ2_history);

p_vol = plot(abs.(R_is), lw=0.5, alpha=0.4, color=:gray, label="|rₜ|",
    title="GARCH(1,1) Conditional Volatility — $(ticker)",
    titlefontsize=10, xlabel="Trading Day", ylabel="Magnitude",
    size=(1000, 400));
plot!(p_vol, cond_vol, lw=2, color=:red, label="GARCH σₜ");
display(p_vol);

savefig(p_vol, joinpath(_PATH_TO_FIGURES, "Fig-$(ticker)-GARCH-CondVol.svg"));

## Simulate N_PATHS Paths

Each simulated path draws i.i.d. Gaussian innovations scaled by the GARCH variance recursion,
starting from the unconditional variance.

In [ ]:
# --- SIMULATE GARCH PATHS ---
n_steps = length(R_is);
garch_sim = Array{Float64,2}(undef, n_steps, N_PATHS);
for i in 1:N_PATHS
    garch_sim[:, i] = simulate_garch(garch_model, n_steps);
end

println("GARCH: $(N_PATHS) paths of length $(n_steps) simulated.");

## Validation Metrics

The same five metrics used for the CHMM in-sample validation are computed here
so the two models can be compared on equal footing:

1. **KS pass rate** (%) at alpha = 0.05
2. **Excess kurtosis** (simulated vs observed)
3. **ACF-MAE** on |G_t| over lags 1-252
4. **Wasserstein-1 distance**
5. **Hellinger distance**

In [ ]:
# --- VALIDATION METRICS FUNCTION ---
function compute_garch_metrics(observed::Vector{Float64}, simulated_archive::Matrix{Float64};
                               α_sig=0.05, L=252)

    n_paths = size(simulated_archive, 2);
    n_obs = length(observed);

    # Observed targets
    μ_obs = mean(observed); σ_obs = std(observed);
    kurt_obs = sum(((observed .- μ_obs) ./ σ_obs).^4) / n_obs - 3.0;
    acf_obs = autocor(abs.(observed), 1:L);

    # Per-path accumulators
    ks_pass = 0;
    kurt_sum = 0.0; acf_mae_sum = 0.0;
    w1_sum = 0.0; hell_sum = 0.0;

    for i in 1:n_paths
        sim = simulated_archive[:, i];

        # KS test
        ks_pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        if ks_pval > α_sig; ks_pass += 1; end

        # Excess kurtosis
        μ_s = mean(sim); σ_s = std(sim);
        kurt_sum += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;

        # ACF-MAE on absolute returns
        acf_sim = autocor(abs.(sim), 1:L);
        acf_mae_sum += mean(abs.(acf_obs .- acf_sim));

        # Wasserstein-1 (sorted quantile difference)
        obs_sorted = sort(observed);
        sim_sorted = sort(sim);
        n_min = min(length(obs_sorted), length(sim_sorted));
        obs_q = [obs_sorted[round(Int, k * length(obs_sorted) / n_min)] for k in 1:n_min];
        sim_q = [sim_sorted[round(Int, k * length(sim_sorted) / n_min)] for k in 1:n_min];
        w1_sum += mean(abs.(obs_q .- sim_q));

        # Hellinger distance (histogram-based)
        edges = range(minimum([observed; sim]) - 10, maximum([observed; sim]) + 10, length=101);
        h_obs = fit(Histogram, observed, edges).weights ./ length(observed);
        h_sim = fit(Histogram, sim, edges).weights ./ length(sim);
        hell_sum += sqrt(sum((sqrt.(h_obs) .- sqrt.(h_sim)).^2)) / sqrt(2);
    end

    # Build results
    ks_rate = round(100 * ks_pass / n_paths, digits=1);
    kurt_sim = round(kurt_sum / n_paths, digits=2);
    kurt_o = round(kurt_obs, digits=2);
    acf_mae = round(acf_mae_sum / n_paths, digits=4);
    w1 = round(w1_sum / n_paths, digits=3);
    hell = round(hell_sum / n_paths, digits=4);

    return (ks_rate=ks_rate, kurtosis_sim=kurt_sim, kurtosis_obs=kurt_o,
            acf_mae=acf_mae, wasserstein=w1, hellinger=hell)
end;

In [ ]:
# --- COMPUTE AND PRINT METRICS ---
metrics = compute_garch_metrics(R_is, garch_sim; L=L);

println("╔══════════════════════════════════════════════════════╗");
println("║  GARCH(1,1) Validation: $ticker                      ║");
println("╠══════════════════════════════════════════════════════╣");
println("║  KS pass rate (α=0.05):  $(lpad(metrics.ks_rate, 8))%              ║");
println("║  Excess kurtosis (sim):  $(lpad(metrics.kurtosis_sim, 8))               ║");
println("║  Excess kurtosis (obs):  $(lpad(metrics.kurtosis_obs, 8))               ║");
println("║  ACF-MAE |Gₜ|:          $(lpad(metrics.acf_mae, 8))               ║");
println("║  Wasserstein-1:          $(lpad(metrics.wasserstein, 8))               ║");
println("║  Hellinger:              $(lpad(metrics.hellinger, 8))               ║");
println("╚══════════════════════════════════════════════════════╝");

## Figure: GARCH(1,1) Benchmark Comparison

Three-panel figure:
- **(a)** Density: histogram of observed returns + density of GARCH simulated returns
- **(b)** ACF(|G_t|): observed vs simulated mean with 10-90th percentile band
- **(c)** Q-Q plot: observed vs simulated quantiles (0.1st to 99.9th percentile)

In [ ]:
# --- FIGURE (a): MARGINAL DENSITY ---
p_a = plot(title="(a) Marginal Density (KS: $(metrics.ks_rate)%)",
    titlefontsize=9, xlabel="Excess Growth Rate", ylabel="Density");

histogram!(p_a, R_is, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");
density!(p_a, vec(garch_sim), lw=2, color=:darkorange, alpha=0.7, label="GARCH(1,1)");
xlims!(p_a, -800, 800);

# --- FIGURE (b): ACF(|G|) COMPARISON ---
τ = 1:L;
acf_obs = autocor(abs.(R_is), τ);

n_acf_paths = min(200, N_PATHS);
acf_archive = hcat([autocor(abs.(garch_sim[:, i]), τ) for i in 1:n_acf_paths]...);
acf_mean = mean(acf_archive, dims=2)[:];
acf_p10 = [quantile(acf_archive[t, :], 0.10) for t in 1:L];
acf_p90 = [quantile(acf_archive[t, :], 0.90) for t in 1:L];

p_b = plot(τ, acf_obs, lw=2, color=:red, ls=:dash, label="Observed",
    title="(b) ACF(|Gₜ|) — Volatility Clustering", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|Gₜ|)");
plot!(p_b, τ, acf_mean, lw=2, color=:darkorange, label="GARCH mean");
plot!(p_b, τ, acf_p10, fillrange=acf_p90, alpha=0.15, color=:darkorange, label="10-90th pctl");

# --- FIGURE (c): TAIL Q-Q PLOT ---
probs_qq = range(0.001, 0.999, length=200);
q_obs = quantile(R_is, probs_qq);
q_sim = quantile(vec(garch_sim), probs_qq);

p_c = plot(q_obs, q_obs, lw=2, color=:black, ls=:dash, label="Perfect",
    title="(c) Tail Q-Q Plot (0.1st-99.9th pctl)", titlefontsize=9,
    xlabel="Observed Quantiles", ylabel="Simulated Quantiles");
scatter!(p_c, q_obs, q_sim, ms=3, alpha=0.6, color=:darkorange, label="GARCH(1,1)");

# --- COMBINE ---
fig_garch = plot(p_a, p_b, p_c, layout=(1, 3), size=(1400, 400),
    plot_title="GARCH(1,1) Benchmark — $(ticker)",
    plot_titlefontsize=12);
display(fig_garch);

savefig(fig_garch, joinpath(_PATH_TO_FIGURES, "Fig-$(ticker)-GARCH-Benchmark.svg"));
savefig(fig_garch, joinpath(_PATH_TO_FIGURES, "Fig-$(ticker)-GARCH-Benchmark.pdf"));

## GARCH(1,1) Limitations

While GARCH(1,1) is a strong benchmark for volatility clustering, it has well-known
limitations that motivate the regime-switching CHMM approach:

1. **Single regime** — GARCH assumes a single set of parameters governs the entire
   sample. It cannot distinguish between structurally different market environments
   (e.g., low-volatility bull markets vs crisis periods). The persistence parameter
   $\alpha + \beta$ is a global average that may oversmooth regime transitions.

2. **Symmetric response** — Standard GARCH(1,1) treats positive and negative shocks
   identically through the $\alpha\,r^2_{t-1}$ term. In practice, negative returns
   increase volatility more than positive returns of the same magnitude (the
   "leverage effect"). Extensions like GJR-GARCH or EGARCH address this, but at the
   cost of additional parameters.

3. **Gaussian innovations** — The model generates returns from $N(\mu, \sigma^2_t)$,
   which produces lighter tails than observed in equity markets. Even with
   time-varying variance, the conditional distribution remains Gaussian. The CHMM,
   by mixing across regimes with different emission parameters, naturally generates
   heavier tails and higher excess kurtosis.

4. **No regime switching** — GARCH cannot capture abrupt transitions between calm
   and crisis states. Volatility must build up gradually through the autoregressive
   mechanism. The CHMM can "jump" between regimes in a single time step, better
   matching the sudden onset of market stress events.

These limitations explain why the CHMM typically outperforms GARCH on metrics like
excess kurtosis matching, tail quantile accuracy (Q-Q), and ACF structure of absolute
returns, despite GARCH having fewer parameters.

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.